# 03. v2 경량 모델 (RDK X5용) — TSM + MobileNetV3-S, 증류, ONNX

- 비디오: **MobileNetV3-Small**(Howard et al. 2019) 2D 백본에 **TSM**(Lin et al. ICCV 2019)
  시간 이동을 삽입 — 추가 연산 0으로 시간 모델링, BPU(Conv/BN/ReLU) 친화적.
- 오디오: 소형 log-mel CNN. 융합 후 11 sigmoid.
- 학습: 라벨 + 02 노트북의 교사 로짓 **지식 증류**(Hinton et al. 2015).
- 내보내기: ONNX(opset 11, 고정 shape) → D-Robotics 툴체인으로 BPU 변환 (마지막 셀 가이드).

In [ ]:
!pip -q install decord av scikit-learn onnx

In [ ]:
# 경로 설정 + Drive 마운트 (Colab)
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE = Path('/content/drive/MyDrive/BabyMon/dataset')   # ← 업로드 위치에 맞게 수정
assert BASE.exists(), f'{BASE} 없음 — Drive 업로드 위치를 확인하세요'

# 클립 폴더 자동 탐색 (clips/ 또는 resized_2/ 또는 BASE 바로 아래)
CLIPS = None
for cand in (BASE/'clips', BASE/'resized_2', BASE):
    if cand.is_dir() and next(cand.glob('*.mp4'), None):
        CLIPS = cand; break
assert CLIPS, f'{BASE} 아래에서 mp4 폴더를 못 찾음 (clips/ 또는 resized_2/)'
assert (BASE/'manifest.csv').exists(), \
    f'{BASE}/manifest.csv 없음 — 로컬 D:\\carved\\dataset\\manifest.csv 와 labels.csv 를 이 폴더로 업로드하세요'
WORK = Path('/content/work'); WORK.mkdir(exist_ok=True)
print('clips:', CLIPS, '/', len(list(CLIPS.glob("*.mp4"))), '개')

In [ ]:
# 02 노트북과 동일한 데이터 준비 (프레임 8개 · 96x160)
import torch, numpy as np, pandas as pd, decord, av, torchaudio
LABEL_COLS = ["D1_nose_covered","D2_moving_freq","D3_eyes_open","D4_hands_out",
              "D5_pre_cry","D6_spit_up","D7_crying","D8_baby_sound","D9_mouthing",
              "C1_adult_hand","C2_baby_absent","C3_other_sound"]
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
T_FR, H_IN, W_IN = 8, 96, 160
BATCH, EPOCHS, LR, KD_W = 16, 20, 6e-4, 0.5
SR, N_MELS = 8000, 64
mel = torchaudio.transforms.MelSpectrogram(SR, n_fft=512, hop_length=256, n_mels=N_MELS)

man = pd.read_csv(BASE/'manifest.csv')
lab = pd.read_csv(BASE/'labels.csv', dtype=str).drop_duplicates('file', keep='last')
df = man.merge(lab, on='file', how='left')
Y = pd.DataFrame(index=df.index, columns=LABEL_COLS, dtype=float)
for c in LABEL_COLS: Y[c] = pd.to_numeric(df.get(c), errors='coerce')
Y.loc[Y['D2_moving_freq'].isna(), 'D2_moving_freq'] = pd.to_numeric(df['weak_D2_moving'], errors='coerce')
Y.loc[Y['D8_baby_sound'].isna(), 'D8_baby_sound'] = pd.to_numeric(df['weak_D8_sound'], errors='coerce')
keep = Y.notna().any(axis=1)
df, Y = df[keep].reset_index(drop=True), Y[keep].reset_index(drop=True)

# 교사 로짓 (없으면 KD 없이 진행)
teacher = None
if (BASE/'teacher_logits.npy').exists():
    tl = np.load(BASE/'teacher_logits.npy')
    tf = pd.read_csv(BASE/'teacher_files.csv')['file'].tolist()
    tmap = {f: tl[i] for i, f in enumerate(tf)}
    teacher = np.stack([tmap.get(f, np.full(len(LABEL_COLS), np.nan)) for f in df['file']])
print('표본', len(df), 'KD', teacher is not None)

def load_video(path):
    vr = decord.VideoReader(str(path))
    ix = np.linspace(0, len(vr)-1, T_FR).astype(int)
    fr = torch.from_numpy(vr.get_batch(ix).asnumpy()).float()/255.
    fr = fr.permute(0,3,1,2)                                          # T,3,H,W
    return torch.nn.functional.interpolate(fr, size=(H_IN, W_IN), mode='bilinear', align_corners=False)

def load_audio(path, sec=8.0):
    try:
        c = av.open(str(path)); s = c.streams.audio[0]
        r = av.audio.resampler.AudioResampler('s16', 'mono', SR)
        buf = [f2.to_ndarray().flatten() for fr in c.decode(s) for f2 in r.resample(fr)]
        x = torch.from_numpy(np.concatenate(buf)).float()/32768.
    except Exception:
        x = torch.zeros(int(SR*sec))
    x = torch.nn.functional.pad(x, (0, max(0, int(SR*sec)-len(x))))[:int(SR*sec)]
    return torch.log(mel(x)+1e-6).unsqueeze(0)

class DS(torch.utils.data.Dataset):
    def __init__(self, sel):
        self.ix = np.where(sel)[0]
    def __len__(self): return len(self.ix)
    def __getitem__(self, k):
        i = self.ix[k]; p = CLIPS/df['file'].iloc[i]
        t = torch.tensor(teacher[i], dtype=torch.float) if teacher is not None else torch.full((len(LABEL_COLS),), float('nan'))
        return load_video(p), load_audio(p), torch.tensor(Y.values[i], dtype=torch.float), t

dl_tr = torch.utils.data.DataLoader(DS((df.split=='train').values), BATCH, shuffle=True, num_workers=2)
dl_va = torch.utils.data.DataLoader(DS((df.split=='val').values), BATCH, shuffle=False, num_workers=2)

In [ ]:
# TSM + MobileNetV3-Small (BPU 친화: Slice/Concat/Conv/BN/ReLU/Pool)
import torchvision

class TSM(torch.nn.Module):
    """채널 1/8 을 앞프레임으로, 1/8 을 뒷프레임으로 이동 (Lin et al. 2019)"""
    def __init__(self, t): super().__init__(); self.t = t
    def forward(self, x):                     # x: (B*T, C, H, W)
        bt, c, h, w = x.shape
        b = bt // self.t
        x = x.view(b, self.t, c, h, w)
        f = c // 8
        left  = torch.cat([x[:, 1:, :f], torch.zeros_like(x[:, :1, :f])], 1)
        right = torch.cat([torch.zeros_like(x[:, :1, f:2*f]), x[:, :-1, f:2*f]], 1)
        x = torch.cat([left, right, x[:, :, 2*f:]], 2)
        return x.reshape(bt, c, h, w)

class LiteAV(torch.nn.Module):
    def __init__(self, n_out=len(LABEL_COLS), t=T_FR):
        super().__init__()
        self.t = t
        mb = torchvision.models.mobilenet_v3_small(weights='IMAGENET1K_V1').features
        self.stem, blocks = mb[:2], list(mb[2:])
        self.blocks = torch.nn.ModuleList()
        for i, blk in enumerate(blocks):      # 4블록마다 TSM 삽입
            if i % 4 == 0:
                self.blocks.append(TSM(t))
            self.blocks.append(blk)
        self.v_fc = torch.nn.Linear(576, 128)
        ch = [1,16,32,64,64]
        self.a_cnn = torch.nn.Sequential(*[torch.nn.Sequential(
            torch.nn.Conv2d(ch[i], ch[i+1], 3, padding=1), torch.nn.BatchNorm2d(ch[i+1]),
            torch.nn.ReLU(), torch.nn.MaxPool2d(2)) for i in range(4)])
        self.a_fc = torch.nn.Linear(64, 64)
        self.head = torch.nn.Sequential(torch.nn.ReLU(), torch.nn.Linear(128+64, n_out))
    def forward(self, v, a):                  # v: (B,T,3,H,W)
        b = v.shape[0]
        x = v.reshape(b*self.t, *v.shape[2:])
        x = self.stem(x)
        for m in self.blocks: x = m(x)
        x = x.mean(dim=(2,3)).view(b, self.t, -1).mean(1)   # 시간 평균
        va = self.v_fc(x)
        aa = self.a_fc(self.a_cnn(a).mean(dim=(2,3)))
        return self.head(torch.cat([va, aa], 1))

model = LiteAV().to(DEV)

def masked_bce(logit, y):
    m = ~torch.isnan(y)
    return torch.nn.functional.binary_cross_entropy_with_logits(logit[m], y[m]) if m.any() else logit.sum()*0

In [ ]:
# 학습: masked BCE + 교사 로짓 증류 (soft BCE)
from sklearn.metrics import average_precision_score
opt = torch.optim.AdamW(model.parameters(), lr=LR)
best = 0
for ep in range(EPOCHS):
    model.train()
    for v, a, y, t in dl_tr:
        v, a, y, t = v.to(DEV), a.to(DEV), y.to(DEV), t.to(DEV)
        logit = model(v, a)
        loss = masked_bce(logit, y)
        mt = ~torch.isnan(t)
        if mt.any():
            loss = loss + KD_W * torch.nn.functional.binary_cross_entropy_with_logits(
                logit[mt], torch.sigmoid(t[mt]))
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval(); P, T = [], []
    with torch.no_grad():
        for v, a, y, _ in dl_va:
            P.append(torch.sigmoid(model(v.to(DEV), a.to(DEV))).cpu()); T.append(y)
    P, T = torch.cat(P).numpy(), torch.cat(T).numpy()
    aps = {c: average_precision_score(T[m,i], P[m,i])
           for i, c in enumerate(LABEL_COLS)
           if (m := ~np.isnan(T[:,i])).sum() and len(set(T[m,i])) > 1}
    mAP = np.mean(list(aps.values())) if aps else 0
    print(f'ep{ep+1}: mAP={mAP:.3f}', {k: round(v,3) for k,v in aps.items()})
    if mAP > best:
        best = mAP; torch.save(model.state_dict(), BASE/'lite_best.pt')
print('best', best)

In [ ]:
# ONNX 내보내기 (opset 11, 고정 shape — BPU 변환 입력)
model.load_state_dict(torch.load(BASE/'lite_best.pt', map_location=DEV)); model.eval().cpu()
v = torch.randn(1, T_FR, 3, H_IN, W_IN)
a = torch.randn(1, 1, N_MELS, 251)
torch.onnx.export(model, (v, a), str(BASE/'lite_av.onnx'), opset_version=11,
                  input_names=['video','audio'], output_names=['logits'])
import onnx; onnx.checker.check_model(str(BASE/'lite_av.onnx'))
print('saved', BASE/'lite_av.onnx')

# BPU 캘리브레이션 세트 (INT8 PTQ용 대표 입력 100개)
import random
cal = WORK/'calib'; cal.mkdir(exist_ok=True)
for i, f in enumerate(random.sample(df['file'].tolist(), min(100, len(df)))):
    np.save(cal/f'v_{i:03d}.npy', load_video(CLIPS/f).unsqueeze(0).numpy())
    np.save(cal/f'a_{i:03d}.npy', load_audio(CLIPS/f).unsqueeze(0).numpy())
print('calibration set →', cal, '(Drive로 복사해 두세요)')

## RDK X5 (BPU) 변환 가이드

Colab이 아니라 **D-Robotics 알고리즘 툴체인 도커**에서 수행합니다
(https://developer.d-robotics.cc/ → Algorithm Toolchain, RDK X5 = BPU **Bayes-e**):

```bash
# 1) 모델 점검
hb_mapper checker --model-type onnx --model lite_av.onnx --march bayes-e
# 2) 변환 (yaml 예시)
hb_mapper makertbin --config convert.yaml --model-type onnx
```
`convert.yaml` 핵심 항목:
```yaml
model_parameters:
  onnx_model: lite_av.onnx
  march: bayes-e
  output_model_file_prefix: lite_av
input_parameters:            # video/audio 두 입력의 shape·범위 지정
  input_name: "video;audio"
  input_shape: "1x8x3x96x160;1x1x64x251"
calibration_parameters:
  cal_data_dir: ./calib      # 위 셀에서 만든 npy
  calibration_type: default
```
- 5차원 video 입력이 문제되면 `(8,3,96,160)` 4차원으로 export 를 바꾸고 모델 안 reshape 를 제거하는 변형이 필요할 수 있음 (BPU 는 NCHW 4D 선호).
- 변환된 `.bin` 은 `hrt_model_exec perf` 로 온보드 지연 측정.